# Лабораторна робота №7 — Тестування в Python
## Частина 1: Загальне тестування та unittest

## 1. Перевірка даних через `assert` та `raise ValueError`

In [1]:
# Перевірка даних через assert
def check_positive_assert(number: float) -> float:
    assert number > 0, "Число має бути більшим за нуль!"
    return number

# Перевірка через raise ValueError
def create_name(name: str) -> str:
    if not name:
        raise ValueError("Ім'я не може бути порожнім")
    return name

# Правильне введення
print("Правильне введення:")
print(check_positive_assert(5))
print(create_name("Іван"))

# Неправильне введення
print("\nНеправильне введення:")
try:
    check_positive_assert(-1)
except AssertionError as e:
    print(f"AssertionError: {e}")

try:
    create_name("")
except ValueError as e:
    print(f"ValueError: {e}")

Правильне введення:
5
Іван

Неправильне введення:
AssertionError: Число має бути більшим за нуль!
ValueError: Ім'я не може бути порожнім


## 2. Функція підрахунку голосних літер

In [2]:
def count_vowels(text: str) -> int:
    """Рахує кількість голосних літер у рядку."""
    vowels = set("aeiouAEIOUаеєиіїоуюяАЕЄИІЇОУЮЯ")
    return sum(1 for ch in text if ch in vowels)

# Тести вручну
print(f'count_vowels("apple") = {count_vowels("apple")}')      # 2
print(f'count_vowels("gym") = {count_vowels("gym")}')          # 0
print(f'count_vowels("") = {count_vowels("")}')                # 0
print(f'count_vowels("12345") = {count_vowels("12345")}')      # 0
print(f'count_vowels("привіт") = {count_vowels("привіт")}')    # 2
print(f'count_vowels("україна") = {count_vowels("україна")}')  # 4

count_vowels("apple") = 2
count_vowels("gym") = 0
count_vowels("") = 0
count_vowels("12345") = 0
count_vowels("привіт") = 2
count_vowels("україна") = 4


## 3. Клас Figure

In [3]:
class Figure:
    VALID_TYPES = ("квадрат", "прямокутник", "трикутник")

    def __init__(self, figure_type: str, length: float):
        if length <= 0:
            raise ValueError("Довжина має бути більшою за нуль")
        if figure_type not in self.VALID_TYPES:
            raise ValueError(f"Невідомий тип фігури: {figure_type}")
        self._figure_type = figure_type
        self._length = length

    @property
    def get_figure_type(self) -> str:
        return self._figure_type

    @property
    def get_figure_length(self) -> float:
        return self._length

    @property
    def get_angles(self) -> int:
        angles = {
            "квадрат": 4,
            "прямокутник": 4,
            "трикутник": 3,
        }
        return angles[self._figure_type]

    def __str__(self) -> str:
        return f"{self._figure_type} зі стороною {self._length}"

# Створюємо кілька об'єктів і перевіряємо вручну
f1 = Figure("квадрат", 5)
f2 = Figure("прямокутник", 3)
f3 = Figure("трикутник", 7)

print(f"Тип: {f1.get_figure_type}, Довжина: {f1.get_figure_length}, Кути: {f1.get_angles}")
print(f"Тип: {f2.get_figure_type}, Довжина: {f2.get_figure_length}, Кути: {f2.get_angles}")
print(f"Тип: {f3.get_figure_type}, Довжина: {f3.get_figure_length}, Кути: {f3.get_angles}")

Тип: квадрат, Довжина: 5, Кути: 4
Тип: прямокутник, Довжина: 3, Кути: 4
Тип: трикутник, Довжина: 7, Кути: 3


## 4. Тести з unittest
### Життєвий цикл: setUp / tearDown

In [4]:
import unittest
from unittest.mock import patch

def ask_user_number() -> str:
    value = input("Введіть число: ")
    return value


class TestValidation(unittest.TestCase):

    def test_assert_positive_number_ok(self):
        self.assertEqual(5, check_positive_assert(5))

    def test_assert_negative_raises(self):
        with self.assertRaises(AssertionError):
            check_positive_assert(-1)

    def test_assert_zero_raises(self):
        with self.assertRaises(AssertionError):
            check_positive_assert(0)

    def test_create_name_ok(self):
        self.assertEqual("Іван", create_name("Іван"))

    def test_create_name_empty_raises(self):
        with self.assertRaises(ValueError):
            create_name("")


class TestCountVowels(unittest.TestCase):

    def test_english_vowels(self):
        self.assertEqual(2, count_vowels("apple"))

    def test_no_vowels(self):
        self.assertEqual(0, count_vowels("gym"))

    def test_all_vowels(self):
        self.assertEqual(3, count_vowels("aei"))

    def test_empty_string(self):
        self.assertEqual(0, count_vowels(""))

    def test_digits_only(self):
        self.assertEqual(0, count_vowels("12345"))

    def test_ukrainian_letters(self):
        self.assertEqual(2, count_vowels("привіт"))
        self.assertEqual(4, count_vowels("україна"))


class TestFigureType(unittest.TestCase):

    def test_square_type(self):
        f = Figure("квадрат", 5)
        self.assertEqual("квадрат", f.get_figure_type)

    def test_rectangle_type(self):
        f = Figure("прямокутник", 3)
        self.assertEqual("прямокутник", f.get_figure_type)

    def test_triangle_type(self):
        f = Figure("трикутник", 7)
        self.assertEqual("трикутник", f.get_figure_type)


class TestFigureLength(unittest.TestCase):

    def setUp(self):
        self.square = Figure("квадрат", 10)

    def tearDown(self):
        del self.square

    def test_length_is_correct(self):
        self.assertEqual(10, self.square.get_figure_length)

    def test_zero_length_raises(self):
        with self.assertRaises(ValueError):
            Figure("квадрат", 0)

    def test_negative_length_raises(self):
        with self.assertRaises(ValueError):
            Figure("квадрат", -5)

    def test_unknown_type_raises(self):
        with self.assertRaises(ValueError):
            Figure("коло", 5)


class TestFigureAngles(unittest.TestCase):

    def test_square_angles(self):
        self.assertEqual(4, Figure("квадрат", 5).get_angles)

    def test_rectangle_angles(self):
        self.assertEqual(4, Figure("прямокутник", 5).get_angles)

    def test_triangle_angles(self):
        self.assertEqual(3, Figure("трикутник", 5).get_angles)


class TestFigureSubTest(unittest.TestCase):

    def test_figure_lengths_subtest(self):
        for length in [1, 5, 10]:
            with self.subTest(length=length):
                f = Figure("квадрат", length)
                self.assertEqual(length, f.get_figure_length)

    def test_all_figure_types_subtest(self):
        for figure_type in ["квадрат", "прямокутник", "трикутник"]:
            with self.subTest(figure_type=figure_type):
                f = Figure(figure_type, 4)
                self.assertEqual(figure_type, f.get_figure_type)


class TestMockInput(unittest.TestCase):

    @patch("builtins.input", return_value="5")
    def test_ask_user_number_returns_value(self, mock_input):
        result = ask_user_number()
        self.assertEqual("5", result)
        mock_input.assert_called_once()

    @patch("builtins.input", return_value="")
    def test_ask_user_number_empty(self, mock_input):
        result = ask_user_number()
        self.assertEqual("", result)


# Запуск тестів у Jupyter Notebook
unittest.main(argv=[""], exit=False, verbosity=2)

test_all_vowels (__main__.TestCountVowels.test_all_vowels) ... ok
test_digits_only (__main__.TestCountVowels.test_digits_only) ... ok
test_empty_string (__main__.TestCountVowels.test_empty_string) ... ok
test_english_vowels (__main__.TestCountVowels.test_english_vowels) ... ok
test_no_vowels (__main__.TestCountVowels.test_no_vowels) ... ok
test_ukrainian_letters (__main__.TestCountVowels.test_ukrainian_letters) ... ok
test_rectangle_angles (__main__.TestFigureAngles.test_rectangle_angles) ... ok
test_square_angles (__main__.TestFigureAngles.test_square_angles) ... ok
test_triangle_angles (__main__.TestFigureAngles.test_triangle_angles) ... ok
test_length_is_correct (__main__.TestFigureLength.test_length_is_correct) ... ok
test_negative_length_raises (__main__.TestFigureLength.test_negative_length_raises) ... ok
test_unknown_type_raises (__main__.TestFigureLength.test_unknown_type_raises) ... ok
test_zero_length_raises (__main__.TestFigureLength.test_zero_length_raises) ... ok
test_all_

## Висновок

- В роботі реалізовано юніт-тестування на Python за допомогою бібліотеки `unittest`
- Написано **25 тестів** для функцій та класу — всі пройшли успішно
- Отримано знання про `assert`, `raise ValueError`, `setUp/tearDown`, `subTest`, `mock patch`
- Знайдено та виправлено помилку: у тесті `test_english_vowels` очікувалось 3 голосні у слові `apple`, але правильне значення — 2
- `subTest` дозволяє перевірити кілька наборів даних в одному тесті і бачити який саме не пройшов
- `mock patch` замінює `input()` під час тесту — не потрібно вводити дані вручну